In [2]:
import jax.numpy as jnp
import roughpy_jax as rpj
import jax
from roughpy_jax.streams import LieIncrementStream
from utils import uniform_intervals, to_list_format, make_incremental
from solver import RoughKernel
import sklearn
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from tslearn.clustering import KernelKMeans
from experiments import perturb_streams

ImportError: DLL load failed while importing _roughpy: An Application Control policy has blocked this file.

In [ ]:
streams_10k = jnp.load("ensembles-10k/ensembles-10k/ensemble-0.npy")

In [ ]:
streams = streams_10k[:200]
Lie_Basis = rpj.LieBasis(width = 4, depth = 2)

In [ ]:
# creating message_A. The shape (25, 10, 10) corresponds to the 25 streams we perturb,
# and the 10 places in which we adjust all 10 Lie values of the stream

message_A = jnp.ones((25, 10, 10), dtype=jnp.float32) * 1e-1

(25, 10, 10)

In [ ]:
# idx_A gives the rows we change in each stream (for now same for every stream)

idx_A = jnp.arange(10)

# sidx_A gives the streams which we will add the message to

sidx_A = jnp.arange(25)

streams_pert = perturb_streams(streams, message_A, Lie_Basis, sidx_A, idx_A)

In [ ]:
times = jnp.arange(0, 2048, dtype=jnp.float32)/2048
times_list = [times for _ in range(200)]
streams_500_list = [streams[b] for b in range(200)]

In [ ]:
streams_LIS = LieIncrementStream.from_increments(
            timestamps=times_list,
            data = streams,
            input_data_basis=Lie_Basis,
            lie_basis=Lie_Basis,
            resolution=10
)

In [ ]:
intervals = uniform_intervals(30)

# instantiating rough kernel

rk = RoughKernel(n=2, R=10)

# pre-computing Gram matrix on data

K = rk.solve_PDE(intervals, streams_LIS, streams_LIS, is_Lie=True)

In [ ]:
tskm = KernelKMeans(n_clusters=2, kernel="precomputed", random_state=0)
ts_labels = tskm.fit_predict(K)

In [ ]:
# y is the variable to be predicted

y = ['N' for _ in range(200)]
for i in sidx_A:
    y[i] = 'A'

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    streams_500_list, 
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
X_train_LIS = LieIncrementStream.from_increments(
            timestamps=times_400,
            data = X_train,
            input_data_basis=Lie_Basis,
            lie_basis=Lie_Basis,
            resolution=10
)
X_test_LIS = LieIncrementStream.from_increments(
            timestamps=times_100,
            data = X_test,
            input_data_basis=Lie_Basis,
            lie_basis=Lie_Basis,
            resolution=10
)

In [ ]:
# choosing intervals is fairly arbitrary at this stage
intervals = uniform_intervals(20)

In [ ]:
# instantiating rough kernel

rk = RoughKernel(n=2, R=10)

# pre-computing Gram matrix on training data

K_train = rk.solve_PDE(intervals, X_train_LIS, X_train_LIS, is_Lie=True)

KeyboardInterrupt: 

In [ ]:
# fitting SVC with the precomputed kernel
svc = SVC(kernel = 'precomputed')
svc.fit(K_train, y_train)

In [ ]:
# precomputing kernel for testing

K_test = rk.solve_PDE(intervals, X_test_LIS, X_train_LIS, is_Lie=True)

# making predictions using our SVC model

y_pred = svc.predict(K_test)

TypeError: RoughKernel.solve_PDE() missing 1 required positional argument: 'is_Lie'

X = streams_500_list
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train_Lie = LieIncrementStream.from_increments(
    timestamps = times_400,
    data = X_train,
    input_data_basis = Lie_Basis,
    lie_basis = Lie_Basis,
    resolution = 10
)
X_test_Lie = LieIncrementStream.from_increments(
    timestamps = times_100,
    data = X_test,
    input_data_basis = Lie_Basis,
    lie_basis = Lie_Basis,
    resolution = 10
)

intervals = uniform_intervals(64)
rk = RoughKernel(n=2, R=10)

K_train = rk.solve_PDE(intervals, X_train, X_train,is_Lie=True)

svc = SVC(kernel='precomputed')
svc.fit(K_train, y_train)

K_test = rk.solve_PDE(intervals, X_test, X_train, is_Lie=is_Lie)
y_pred = svc.predict(K_test)

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

In [ ]:
import jax.numpy as jnp
import roughpy_jax as rpj
import jax
from roughpy_jax.streams import LieIncrementStream
from utils import uniform_intervals
from solver import RoughKernel
import sklearn
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

N = 2000 # no. of streams
K = 50 # no. of streams to change
M = 10 # no. increments to change in each stream
I = 50 # no. of intervals


streams_10k = jnp.load('ensembles-10k/ensembles-10k/ensemble-0.npy')

streams = streams_10k[:N]
Lie_Basis = rpj.LieBasis(width = 4, depth = 2)

# creating message_A. The shape (K, M, 10) corresponds to the K streams we perturb,
# and the M places in which we adjust all 10 Lie values of the stream

message_A = jnp.ones((K, M, 10), dtype=jnp.float32) * 1e-1
message_A_Lie = rpj.Lie(message_A, Lie_Basis)

# idx_A gives the rows we change in each stream (for now same for every stream)

idx_A = jnp.array([200 * i for i in range(M)])

# sidx_A gives the streams which we will add the message to

sidx_A = jnp.array([i*20 for i in range(K)])

# grid0 and grid1 ease changing of arrays later in JAX

grid0, grid1 = jnp.ix_(sidx_A, idx_A)

# perturbing the streams (in streams and places decided aboves) using cbh

pieces_to_perturb = streams[grid0, grid1]
pieces_to_perturb_Lie = rpj.Lie(pieces_to_perturb, Lie_Basis)
perturbed_pieces = rpj.algebra.cbh(pieces_to_perturb_Lie, message_A_Lie).data

streams_pert = streams.at[grid0, grid1].set(perturbed_pieces)

times = jnp.arange(0, 2048, dtype=jnp.float32)/2048
times_train = [times for _ in range(4*N // 5)]
times_test = times_train[:(N//5)]
streams_plist = [streams_pert[b] for b in range(N)]

# y is the variable to be predicted

y = ['N' for _ in range(N)]
for i in sidx_A:
    y[i] = 'A'

X_train, X_test, y_train, y_test = train_test_split(
    streams_plist, 
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)   
X_train_LIS = LieIncrementStream.from_increments(
            timestamps=times_train,
            data = X_train,
            input_data_basis=Lie_Basis,
            lie_basis=Lie_Basis,
            resolution=10
)
X_test_LIS = LieIncrementStream.from_increments(
            timestamps=times_test,
            data = X_test,
            input_data_basis=Lie_Basis,
            lie_basis=Lie_Basis,
            resolution=10
)

# choosing intervals is fairly arbitrary at this stage
intervals = uniform_intervals(I)

# instantiating rough kernel

rk = RoughKernel(n=2, R=10)

In [ ]:
print((1, 2))

(1, 2)
